# YOLO11n-Pose Training — COCO 2017
**Posyandu Bayi Height Measurement Project**

### Cara Pakai:
1. Buka di Google Colab: `File → Upload notebook` atau drag file ini
2. Pastikan GPU aktif: `Runtime → Change runtime type → T4 GPU`
3. Jalankan sel satu per satu dari atas ke bawah
4. Kalau sesi disconnect, jalankan ulang dari **Sel 1** (mount Drive) lalu langsung ke **Sel 6** (resume training)

### Estimasi Waktu (GPU T4):
- 1 epoch ≈ 3-5 menit
- 50 epoch ≈ 2-4 jam

---
## Sel 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Folder utama di Google Drive kamu
DRIVE_DIR = '/content/drive/MyDrive/posyandu_yolo'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Working dir:', DRIVE_DIR)

---
## Sel 2 — Install Dependencies

In [ ]:
!pip install ultralytics -q
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: GPU tidak aktif! Pergi ke Runtime → Change runtime type → T4 GPU')

---
## Sel 3 — Download COCO 2017 Dataset
> **Lewati sel ini** kalau dataset sudah ada di Drive dari sesi sebelumnya.
> Cek dulu dengan menjalankan sel cek di bawah.

In [ ]:
import os

DATASET_DIR = f'{DRIVE_DIR}/coco2017'
train_images = f'{DATASET_DIR}/train2017'
val_images   = f'{DATASET_DIR}/val2017'
ann_dir      = f'{DATASET_DIR}/annotations'

# Cek apakah dataset sudah ada
train_count = len(os.listdir(train_images)) if os.path.exists(train_images) else 0
val_count   = len(os.listdir(val_images))   if os.path.exists(val_images)   else 0
ann_exists  = os.path.exists(f'{ann_dir}/person_keypoints_train2017.json')

print(f'Train images : {train_count:,}')
print(f'Val images   : {val_count:,}')
print(f'Annotations  : {ann_exists}')

if train_count < 100000 or not ann_exists:
    print('\nDataset belum lengkap. Jalankan sel download di bawah.')
else:
    print('\nDataset sudah ada! Lewati sel download.')

In [ ]:
# ============================================================
# JALANKAN HANYA JIKA DATASET BELUM ADA DI DRIVE
# Download ~20 GB, estimasi 20-30 menit di Colab
# ============================================================
import os
DATASET_DIR = f'{DRIVE_DIR}/coco2017'
os.makedirs(DATASET_DIR, exist_ok=True)
os.chdir(DATASET_DIR)

# Download images
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# Extract
print('Extracting train2017...')
!unzip -q train2017.zip
print('Extracting val2017...')
!unzip -q val2017.zip
print('Extracting annotations...')
!unzip -q annotations_trainval2017.zip

# Cleanup zips untuk hemat space
!rm train2017.zip val2017.zip annotations_trainval2017.zip
print('Download selesai!')

---
## Sel 4 — Konversi Anotasi COCO → YOLO Format
> **Lewati sel ini** kalau label `.txt` sudah ada dari sesi sebelumnya.

In [ ]:
import os
DATASET_DIR = f'{DRIVE_DIR}/coco2017'
label_train = f'{DATASET_DIR}/train2017_labels'

label_count = len(os.listdir(label_train)) if os.path.exists(label_train) else 0
print(f'Label files existing: {label_count:,}')
if label_count > 50000:
    print('Labels sudah ada! Lewati konversi.')
else:
    print('Perlu konversi. Jalankan sel di bawah.')

In [ ]:
# ============================================================
# KONVERSI COCO JSON → YOLO TXT (jalankan 1x saja)
# Estimasi: ~5-10 menit
# ============================================================
import json
from pathlib import Path
from collections import defaultdict

DATASET_DIR = f'{DRIVE_DIR}/coco2017'
NUM_KP = 17

def convert_split(split_name, ann_json, img_dir, label_dir):
    Path(label_dir).mkdir(parents=True, exist_ok=True)
    ann_file = f'{DATASET_DIR}/annotations/{ann_json}'
    print(f'Reading {ann_json}...')
    with open(ann_file) as f:
        data = json.load(f)

    img_lookup = {img['id']: img for img in data['images']}
    ann_by_img = defaultdict(list)

    for ann in data['annotations']:
        if ann.get('iscrowd', 0): continue
        kps = ann.get('keypoints', [])
        if len(kps) != NUM_KP * 3: continue
        if not any(kps[i*3+2] == 2 for i in range(NUM_KP)): continue
        ann_by_img[ann['image_id']].append(ann)

    written = 0
    for image_id, anns in ann_by_img.items():
        img_info = img_lookup.get(image_id)
        if not img_info: continue
        img_path = f'{img_dir}/{img_info["file_name"]}'
        if not Path(img_path).exists(): continue

        W, H = img_info['width'], img_info['height']
        lines = []
        for ann in anns:
            bx, by, bw, bh = ann['bbox']
            cx = (bx + bw/2) / W
            cy = (by + bh/2) / H
            bw_n = bw / W
            bh_n = bh / H
            kps = ann['keypoints']
            kp_str = ' '.join(
                f'{kps[i*3]/W:.6f} {kps[i*3+1]/H:.6f} {kps[i*3+2]}'
                for i in range(NUM_KP)
            )
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw_n:.6f} {bh_n:.6f} {kp_str}')

        stem = Path(img_info['file_name']).stem
        with open(f'{label_dir}/{stem}.txt', 'w') as f:
            f.write('\n'.join(lines))
        written += 1

    print(f'  {split_name}: {written:,} label files written')
    return written

convert_split('train', 'person_keypoints_train2017.json',
              f'{DATASET_DIR}/train2017', f'{DATASET_DIR}/train2017_labels')
convert_split('val',   'person_keypoints_val2017.json',
              f'{DATASET_DIR}/val2017',   f'{DATASET_DIR}/val2017_labels')
print('Konversi selesai!')

---
## Sel 5 — Buat data.yaml

In [ ]:
import os
DATASET_DIR = f'{DRIVE_DIR}/coco2017'

yaml_content = f"""# YOLO11-Pose COCO 2017 — Posyandu Project
path: {DATASET_DIR}

train: train2017
val:   val2017

# Label dirs (YOLO akan auto-detect dari path images → labels)
# train labels: {DATASET_DIR}/train2017_labels
# val labels:   {DATASET_DIR}/val2017_labels

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

nc: 1
names:
  0: person
"""

yaml_path = f'{DRIVE_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('data.yaml saved to:', yaml_path)
print(yaml_content)

---
## Sel 5b — Symlink Labels ke Samping Images
> YOLO mencari label di folder bernama `labels` yang sejajar dengan `images`.
> Karena kita simpan label terpisah, perlu dibuat symlink.

In [ ]:
import os
DATASET_DIR = f'{DRIVE_DIR}/coco2017'

# Copy labels ke dalam folder images (paling mudah, tanpa perlu symlink)
# Ini hanya perlu dijalankan 1x — akan skip file yang sudah ada
import shutil
from pathlib import Path

def copy_labels_to_imgdir(label_dir, img_dir):
    label_files = list(Path(label_dir).glob('*.txt'))
    print(f'Copying {len(label_files):,} label files to {img_dir}...')
    for lf in label_files:
        dest = Path(img_dir) / lf.name
        if not dest.exists():
            shutil.copy2(lf, dest)
    print('Done!')

copy_labels_to_imgdir(f'{DATASET_DIR}/train2017_labels', f'{DATASET_DIR}/train2017')
copy_labels_to_imgdir(f'{DATASET_DIR}/val2017_labels',   f'{DATASET_DIR}/val2017')

---
## Sel 6 — Mulai / Lanjutkan Training
> **Kalau sesi disconnect**, langsung jalankan **Sel 1** (mount Drive) lalu sel ini dengan `resume=True`.

In [ ]:
import os
from ultralytics import YOLO

DRIVE_DIR    = '/content/drive/MyDrive/posyandu_yolo'
RESULTS_DIR  = f'{DRIVE_DIR}/runs'
YAML_PATH    = f'{DRIVE_DIR}/data.yaml'
LAST_CKPT    = f'{RESULTS_DIR}/pose/coco_pose_v1/weights/last.pt'

# Auto-detect: resume kalau checkpoint ada, mulai baru kalau belum
if os.path.exists(LAST_CKPT):
    print(f'Checkpoint ditemukan: {LAST_CKPT}')
    print('Melanjutkan training...')
    model = YOLO(LAST_CKPT)
    model.train(resume=True)
else:
    print('Mulai training baru...')
    model = YOLO('yolo11n-pose.pt')  # download otomatis
    model.train(
        data=YAML_PATH,
        epochs=10,         # testing dulu 10 epoch (~30 menit di T4)
        batch=32,          # batch 32 untuk GPU T4
        imgsz=640,
        device=0,          # GPU
        project=RESULTS_DIR,
        name='coco_pose_v1',
        exist_ok=True,
        workers=4,
        save_period=2,     # simpan checkpoint setiap 2 epoch
        patience=10,
    )

---
## Sel 7 — Evaluasi & Export Model

In [ ]:
from ultralytics import YOLO
import os

DRIVE_DIR   = '/content/drive/MyDrive/posyandu_yolo'
BEST_MODEL  = f'{DRIVE_DIR}/runs/pose/coco_pose_v1/weights/best.pt'

model = YOLO(BEST_MODEL)

# Evaluasi
metrics = model.val()
print('mAP50-95 (pose):', metrics.pose.map)
print('mAP50 (pose)   :', metrics.pose.map50)

# Export ke format TFLite untuk Flutter/Android
model.export(
    format='tflite',
    imgsz=640,
    int8=False,    # True untuk model lebih kecil (butuh kalibrasi data)
)
print('Export selesai!')
print('File tersimpan di:', os.path.dirname(BEST_MODEL))

---
## Sel 8 — Download Model ke Lokal
> Download file `.pt` dan `.tflite` langsung ke komputer.

In [ ]:
from google.colab import files

DRIVE_DIR  = '/content/drive/MyDrive/posyandu_yolo'
WEIGHTS_DIR = f'{DRIVE_DIR}/runs/pose/coco_pose_v1/weights'

# Download best.pt
files.download(f'{WEIGHTS_DIR}/best.pt')

# Download TFLite (kalau sudah di-export)
import glob
tflite_files = glob.glob(f'{WEIGHTS_DIR}/*.tflite')
for f in tflite_files:
    files.download(f)
    print('Downloaded:', f)